# 日内 Value–Weight 信息增量筛查

依据华泰文章，适配美国完整分钟路径。固定18次训练已完成；所有区间是已复用历史。
[冻结设计](plan.json) · [独立设计评审](design_review.json) · [信息指标](information_metrics.csv) · [年度/季度](regime_window_metrics.csv) · [不确定性](uncertainty.csv)

Qlib D.features / DatasetH 管理输入，Recorder保存模型与预测。由于交易对象只有一个SOXL相对SOXX时序标签，使用跨日期Spearman与配对MSE分析；不调用退化的两资产横截面IC，不假造投资组合报告。

In [1]:
from pathlib import Path
import os
os.environ["MLFLOW_ALLOW_FILE_STORE"]="true"
import json, qlib
from qlib.data import D
from qlib.workflow import R
root=Path.cwd()
while not (root/'SPEC.md').exists(): root=root.parent
import sys
sys.path.insert(0,str(root))
from src.workflow.minute_value_weight import run
card=root/'research/minute_value_weight_20261017'
plan=json.loads((card/'plan.json').read_text())
qlib.init(provider_uri=str(root/plan['providers']['daily']),region='us',kernels=1)
summary=run(root=root,card=card,plan=plan)
summary

2026/10/17 09:27:16 INFO mlflow.agent.hint: Load the `instrumenting-with-mlflow-tracing` skill at /Users/rziiiii/dev/workspace/qlib_venv/lib/python3.12/site-packages/mlflow/assistant/skills/instrumenting-with-mlflow-tracing/SKILL.md before writing any tracing code; it ships with this MLflow install. Set MLFLOW_DISABLE_AGENT_HINT=1 to silence this.


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


[59826:MainThread](2026-10-17 09:27:17,601) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[59826:MainThread](2026-10-17 09:27:17,603) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[59826:MainThread](2026-10-17 09:27:17,603) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/data/qlib/minute_information_pit_allocator_20261012/daily_prices')}


{'experiment': 'minute_value_weight_20261017',
 'plan_sha256': '202a0d38175ea113f553e4396f52c5c4c356e24b9ed6513e163e95f6db183dc0',
 'decision': 'not_promoted_information_gate_failed',
 'objective_achieved': False,
 'gates': {'development_2021_2022_positive_spearman_ic': True,
  'development_2021_2022_mse_better_handcrafted': False,
  'development_2021_2022_two_seeds_handcrafted': False,
  'development_2021_2022_mse_better_uniform': True,
  'development_2021_2022_two_seeds_uniform': True,
  'reused_2023_2026_positive_spearman_ic': True,
  'reused_2023_2026_mse_better_handcrafted': True,
  'reused_2023_2026_two_seeds_handcrafted': True,
  'reused_2023_2026_mse_better_uniform': True,
  'reused_2023_2026_two_seeds_uniform': True,
  'paired_bootstrap_handcrafted': False,
  'paired_bootstrap_uniform': False},
 'new_fits': 18,
 'new_native_cases': 0,
 'qualified_forward_days': 0,
 'history_status': 'reused_diagnostic_not_independent',
 'as_of': '2026-10-09',
 'provider_uri': '/Users/rziiiii/d

In [2]:
# 只读已完成的训练证据，补充无拟合常数基准；不改变冻结门槛或重训。
import numpy as np
import pandas as pd
import hashlib
from src.data.nas_market_data import archive_nas_recorder
from src.data.minute_intake import atomic_json
rows=[]
from src.workflow.minute_value_weight import prepare
import pickle
# Repair serialization only; verify source identities and do not fit models.
dataset,coverage,audit=prepare(root,plan)
replay=pickle.loads(pickle.dumps(dataset))
assert replay.prepare('all').equals(dataset.prepare('all'))
with R.start(experiment_name=plan['experiment'],uri=summary['recorder_uri'],recorder_id=summary['recorder_id'],resume=True):
    rec=R.get_recorder()
    rec.save_objects(artifact_path='inputs',dataset=dataset,audit=audit)
    rec.save_objects(artifact_path='sources',serialization_repair_workflow=(root/'src/workflow/minute_value_weight.py').read_text())
    ensemble=rec.load_object('analysis/ensemble')
    for fold in plan['folds']:
        name=fold['name']; state=rec.load_object(f"models/{name}_{plan['seeds'][0]}_handcrafted")
        label=ensemble.loc[(ensemble.fold==name)&(ensemble.family=='learned'),'label']
        for baseline,value in [('training_mean',state['label_mean']),('zero_excess',0.)]:
            rows.append({'fold':name,'baseline':baseline,'prediction':float(value),'n':len(label),'mse':float(np.mean((label-value)**2))})
    constants=pd.DataFrame(rows)
    rec.save_objects(artifact_path='analysis',constant_baseline_diagnostic=constants)
    rec.save_objects(artifact_path='sources',finalization_notebook_cell='No-fit constant baseline from saved training scaler; no selection or gate changes.')
    rec.log_metrics(registered_artifact_count=sum(p.is_file() for p in (Path(summary['recorder_local_dir'])/'artifacts').rglob('*')))
constants.to_csv(card/'constant_baseline_diagnostic.csv',index=False)
summary['dataset_serialization_verified']={'shape':list(replay.prepare('all').shape),'all_data_roundtrip_equal':True,'model_refits':0}
summary['constant_baseline_status']='post-run descriptive diagnostic only; no fit, gate or selection changes'
summary['nas_evidence']=archive_nas_recorder(card/'nas_binding.json',workspace_root=root,card=card,recorder_id=summary['recorder_id'],experiment_id=summary['experiment_id'])
atomic_json(summary,card/'summary.json')
constants


[59826:MainThread](2026-10-17 09:27:17,748) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[59826:MainThread](2026-10-17 09:27:17,749) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[59826:MainThread](2026-10-17 09:27:17,749) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/data/qlib/minute_information_pit_allocator_20261012/core_1min')}


[59826:MainThread](2026-10-17 09:27:29,051) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[59826:MainThread](2026-10-17 09:27:29,052) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


...

[59826:MainThread](2026-10-17 09:27:29,129) INFO - qlib.workflow - [exp.py:258] - Experiment 144888956312247735 starts running ...


[59826:MainThread](2026-10-17 09:27:29,135) INFO - qlib.workflow - [recorder.py:345] - Recorder 0f1a94b799624d258aa1bf8acd18044d starts running under Experiment 144888956312247735 ...


Exception in thread Thread-5 (run):
Traceback (most recent call last):
  File "/Users/rziiiii/dev/workspace/qlib_venv/lib/python3.12/site-packages/mlflow/tracking/_tracking_service/client.py", line 413, in log_param
    self.store.log_param(run_id, param)
  File "/Users/rziiiii/dev/workspace/qlib_venv/lib/python3.12/site-packages/mlflow/store/tracking/file_store.py", line 1130, in log_param
    self._log_run_param(run_info, param)
  File "/Users/rziiiii/dev/workspace/qlib_venv/lib/python3.12/site-packages/mlflow/store/tracking/file_store.py", line 1136, in _log_run_param
    self._validate_new_param_value(
  File "/Users/rziiiii/dev/workspace/qlib_venv/lib/python3.12/site-packages/mlflow/store/tracking/file_store.py", line 1156, in _validate_new_param_value
    raise MlflowException(
mlflow.exceptions.MlflowException: Changing param values is not allowed. Param with key='cmd-sys.argv' was already logged with value='/Users/rziiiii/dev/workspace/qlib_venv/lib/python3.12/site-packages/ipy

[59826:MainThread](2026-10-17 09:27:30,578) INFO - qlib.timer - [log.py:127] - Time cost: 0.000s | waiting `async_log` Done


,fold,baseline,prediction,n,mse
0,development_2021_2022,training_mean,0.013498,503,0.008945
1,development_2021_2022,zero_excess,0.000000,503,0.008712
2,reused_2023_2026,training_mean,0.006184,940,0.010062
3,reused_2023_2026,zero_excess,0.000000,940,0.010218


## 中文结论

已执行原始完整分钟路径的 Value MLP + Weight GRU 适配筛查，比较人工12项摘要、均匀聚合、学习时间权重；3个seed、2个时间切分，共18次固定20epoch训练。Qlib有效完整分钟日2007个（其中17个半日市），有标签的评估日为2021–2022的503日、2023–2026-10的940日。H5标签在完整日历上先移位，训练严格过滤尚未成熟标签。

| 区间 | 人工摘要IC | 均匀聚合IC | 学习权重IC |
|---|---:|---:|---:|
| 2021–2022 | 0.0630 | 0.0545 | 0.0551 |
| 2023–2026-10 | 0.0102 | 0.0370 | 0.0370 |

学习权重相对均匀聚合的MSE仅降低约0.033%和0.034%；相对人工摘要在前段反而差0.51%，后段改善0.89%。配对63日区块检验两项下界均不为正（对人工摘要−2.15e−5，对均匀聚合−4.02e−7），前段对人工摘要仅1/3 seed改善。年度/季度全部列于regime_window_metrics.csv，126个有效观测日滚动结果位于Recorder analysis/intervals；2026为不完整年度，跨缺失日的窗口不能称为126连续交易日。

**结论：not_promoted_information_gate_failed。** 这次未证明学习分钟权重有稳定预测增量，固定版本到此关闭。未执行投资组合账户，不能据此宣称净收益、回撤改善或跑赢SOXX；费用与30%最大回撤等经济门槛尚未评估。

所有历史已复用，没有独立前向资格。学习权重还增加GRU容量，不能解释为纯权重的容量匹配证明。文章横截面股票任务已改为两ETF时序H5预测，因此不是完整复现。记录常数基准仅帮助判断预测误差，属于训练结束后描述性对照，不改变原门槛。

下一步需要新的可证伪机制或独立数据证据，不能对该固定版本继续择窗、择seed、改阈值救结果；这个负结果不代表所有原始分钟信息都无效。若要检验文章原始横截面机制，具体需要历史时点合格的半导体成分股名单、这些成分股同步的分钟OHLCV与对应未来收益标签；当前两ETF时序任务不能评估原文横截面BN/Corr主张。本轮不新增采集或训练，也不承诺该机制会成功。

Qlib Recorder：`0f1a94b799624d258aa1bf8acd18044d`，NAS对象身份见nas_binding.json。